#1.不等式制約編
　このノートブックでは、量子アニーリングで不等式制約を扱う方法を学びます。具体的には、「容量制限のあるナップサック問題」を例に、数式（ハミルトニアン）をPythonコード（QUBO行列）に落とし込む過程をステップ・バイ・ステップで実装します。

##2.問題定義

###2.1.ナップサック問題とは？

* $N$ 個の商品があります。
* 各商品 $i$ には、重さ $w_i$ と 価値 $v_i$ が設定されています。
* ナップサックの容量は $C$ です。  
**目標:** 重さの合計が容量 $C$ を超えない範囲で、ナップサックに入れた商品の価値の合計を最大化すること。

###2.2.標準的な定式化

この問題は、数学的には以下のように記述されます。

$$
\begin{aligned}
& \text{Maximize} & & \sum_{i=0}^{N-1} v_i x_i \\
& \text{subject to} & & \sum_{i=0}^{N-1} w_i x_i \le C \\
& & & x_i \in \{0, 1\}
\end{aligned}
$$

###2.3.変数の説明

この数式に登場する記号の意味は以下の通りです。

| 記号 | 名称 | 説明 |
| :--- | :--- | :--- |
| **$N$** | 商品の数 | 全体でいくつの商品があるか。 |
| **$i$** | インデックス | 商品の番号 ($0, 1, \dots, N-1$)。 |
| **$x_i$** | **決定変数** | 最適化によって求めたい変数。<br>・$1$: 商品 $i$ をナップサックに**入れる**<br>・$0$: 商品 $i$ をナップサックに**入れない** |
| **$v_i$** | 価値 (Value) | 商品 $i$ の価値（定数）。この合計を最大化したい。 |
| **$w_i$** | 重さ (Weight) | 商品 $i$ の重さ（定数）。 |
| **$C$** | 容量 (Capacity) | ナップサックに入る最大の重さ（定数）。 |

##3.数理モデルの構築(不等式から等式への変換)

###3.1.目的関数
「価値の最大化」は「マイナスの価値の最小化」と同じです。

$$H_{obj} = - \sum_{i=0}^{N-1} v_i x_i$$

###3.2.制約条件（不等式の処理）
「重さの合計 $\le C$」という不等式制約は、そのままでは扱いにくいです。そこで、「実際にナップサックに入れた重さが $k$ kg である」ことを表す補助変数$y_k$ を導入して、等式制約に変換します。

ここで、$k$ は $1, 2, \dots, C$ の値を取り、補助変数 $y_1, y_2, \dots, y_C$ は**どれか1つだけが1になる（One-hot）必要があります。

制約は以下の2つのペナルティ項として表現されます。

1.  **One-hot制約 ($H_{cost}^{(1)}$):** 補助変数 $y_k$ のうち、必ず1つだけが1になる。
$$H_{cost}^{(1)} = \lambda \left( 1 - \sum_{k=1}^{C} y_k\right)^2$$
2.  **重量一致制約 ($H_{cost}^{(2)}$):** 選ばれた補助変数が示す重さ $k$ と、実際の商品重さの合計が一致する。
$$H_{cost}^{(2)} = \lambda \left( \sum_{k=1}^{C} k y_k - \sum_{i=0}^{N-1} w_i x_i \right)^2$$

（$\lambda$ はペナルティの強さを表す大きな正の定数です）

###3.3.全体のハミルトニアン

これらを合計したものが、最終的に最小化すべきエネルギー関数です。

$$H = H_{obj} + H_{cost}^{(1)} + H_{cost}^{(2)}$$
$$H = - \sum_{i} v_i x_i + \lambda \left( 1 - \sum_{k} y_k \right)^2 + \lambda \left( \sum_{k} k y_k - \sum_{i} w_i x_i \right)^2$$

##4.ハミルトニアンのQUBO式への変形

プログラムで実装するには、ハミルトニアンを $x_i x_j$ や $y_k y_l$ といった変数の積の形（QUBO形式）に展開する必要があります。
バイナリ変数の性質 $x^2 = x$ を利用して展開します。

###4.1.目的関数$H_{obj}$

これはすでに単純な形なので、そのまま対角成分$-v_i$になります。
$$H_{obj} = \sum_{i} (-v_i) x_i$$

###4.2.制約項1 $H_{cost}^{(1)}$ (One-hot制約)

式：$\lambda ( 1 - \sum_{k} y_k )^2$
\begin{aligned}
H_{cost}^{(1)} &= \lambda \left( 1 - 2\sum_{k} y_k + \left(\sum_{k} y_k\right)^2 \right) \\
&= \lambda \left( 1 - 2\sum_{k} y_k + \left( \sum_{k} y_k^2 + \sum_{k \neq l} y_k y_l \right) \right) \\
&\quad \text{※ バイナリ変数の性質 } y_k^2 = y_k \text{ を適用} \\
&= \lambda \left( 1 - 2\sum_{k} y_k + \sum_{k} y_k + 2\sum_{k < l} y_k y_l \right) \\
&= \lambda \left( 1 - \sum_{k} y_k + 2\sum_{k < l} y_k y_l \right) \\
&\simeq \sum_{k} (-\lambda) y_k + \sum_{k < l} (2\lambda) y_k y_l \quad \text{(定数項は無視)}
\end{aligned}

###4.3.制約項2 $H_{cost}^{(2)}$ (重量一致制約)

ここが最も複雑な部分です。式を3つのパートに分けて展開します。
$$H_{cost}^{(2)} = \lambda \left( \underbrace{\sum_{k=1}^{C} k y_k}_{A} - \underbrace{\sum_{i=0}^{N-1} w_i x_i}_{B} \right)^2 = \lambda(A^2 - 2AB + B^2)$$

#### **Part A: 補助変数の2乗項 ($A^2$)**
$$
\begin{aligned}
\lambda \left(\sum_{k} k y_k\right)^2 &= \lambda \left( \sum_{k} k^2 y_k + 2\sum_{k < l} k l y_k y_l \right)
\end{aligned}
$$
行列 $Q$ への対応:
* **$Q_{yy}$ (対角):** $\lambda k^2$
* **$Q_{yl}$ (交差 $k<l$):** $2 \lambda k l$

#### **Part B: 商品変数の2乗項 ($B^2$)**
$$
\begin{aligned}
\lambda \left(\sum_{i} w_i x_i\right)^2 &= \lambda \left( \sum_{i} w_i^2 x_i + 2\sum_{i < j} w_i w_j x_i x_j \right)
\end{aligned}
$$
行列 $Q$ への対応:
* **$Q_{ii}$ (対角):** $\lambda w_i^2$
* **$Q_{ij}$ (交差 $i<j$):** $2 \lambda w_i w_j$

#### **Part C: 交差項 ($-2AB$)**
$$
\begin{aligned}
-2\lambda \left(\sum_{k} k y_k\right) \left(\sum_{i} w_i x_i\right) &= -2\lambda \sum_{k}\sum_{i} k w_i y_k x_i
\end{aligned}
$$
行列 $Q$ への対応:
* **$Q_{yi}$ (補助変数と商品変数の交差):** $-2 \lambda k w_i$

##5.Pythonによる実装とOpenjijでの解法

###5.1.使用するライブララリのインストール

In [2]:
pip install openjij numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 12.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


###5.2.ライブラリのインポート

In [3]:
import numpy as np
import openjij as oj

###5.3.問題の設定

In [4]:
# ==========================================
# 問題データの設定 (Problem Setup)
# ==========================================
# 荷物のリストを定義します。
# 各辞書は1つの商品を表現しており、キーの意味は以下の通りです:
#   'weight': 数式の w_i (重さ)
#   'value':  数式の v_i (価値)
items = [
    {'weight': 2, 'value': 3},
    {'weight': 3, 'value': 4},
    {'weight': 4, 'value': 5},
    {'weight': 5, 'value': 8}, # 重い(5kg)けれど、価値が高い(8)アイテム。これを選ぶかがポイント。
    {'weight': 1, 'value': 1}
]

# 計算しやすいように、重さと価値をそれぞれ別のリストに分離します
# weights[i] で i番目の重さ、values[i] で i番目の価値にアクセスできるようにするためです
weights = [item['weight'] for item in items]
values = [item['value'] for item in items]

# ==========================================
# 定数・パラメータの定義 (Parameters)
# ==========================================
N = len(items)       # 商品の総数 (数式の N)
                     # -> これにより、商品変数 x_0 ... x_{N-1} が必要になります

C = 7                # ナップサックの容量制限 (数式の C)
                     # -> 重さの合計がこれ以下でなければなりません (Σw_i x_i <= C)

# ペナルティ係数 (数式の λ: Lambda)
# 制約（容量オーバーやOne-hot違反）を破った場合に課される罰則の強さです。
# もしこの値が小さすぎると、「ペナルティを受けてでも価値が高い商品を入れたほうが得」
# と判断されてしまい、ルール違反の解が出てしまいます。
# そのため、商品の価値よりも十分に大きな値 (ここでは100.0) を設定します。
LAMBDA = 100.0

# 設定内容の確認表示
print(f"Items (Weight, Value): {list(zip(weights, values))}")
print(f"Capacity: {C}")

Items (Weight, Value): [(2, 3), (3, 4), (4, 5), (5, 8), (1, 1)]
Capacity: 7


###5.4.QUBO行列の作成

In [7]:
# ==========================================
# QUBO行列の初期化 (ゼロ行列)
# ==========================================
# N+C × N+C の真っ新な行列を作る
Q = np.zeros((N+C, N+C))

# --- インデックス操作用の便利関数 ---
def get_q_idx(i):
    """商品 i (0~N-1) の行列内のインデックス"""
    return i

def get_y_idx(k):
    """補助変数 y_k (k=1~C) の行列内のインデックス"""
    # 商品変数の後ろに続く。kは1始まりなので -1 して 0始まりに合わせる
    return N + (k - 1)

# ==========================================
# QUBO行列への埋め込み
# ==========================================

# ------------------------------------------------
# (A) 目的関数: 価値の最大化 (-価値の最小化)
# ------------------------------------------------
for i in range(N):
    idx = get_q_idx(i)
    v = items[i]['value']
    # 対角成分に -v を加算
    Q[idx, idx] += -1 * v

# ------------------------------------------------
# (B) 制約項1: 補助変数 y はどれか1つだけ (One-hot)
# 数式: lambda * (1 - sum(y))^2
# 展開 -> lambda * ( sum(y^2) + sum(y_i*y_j) - 2sum(y) ) + 定数
# ------------------------------------------------
for k1 in range(1, C + 1):
    idx1 = get_y_idx(k1)

    # 1. 線形項: -2 * lambda * y_k
    Q[idx1, idx1] += -2 * LAMBDA

    # 2. 二乗項の対角成分: + lambda * y_k^2 (= y_k)
    Q[idx1, idx1] += LAMBDA

    # 3. 二乗項の交差成分: + 2 * lambda * y_k1 * y_k2
    for k2 in range(k1 + 1, C + 1):
        idx2 = get_y_idx(k2)
        Q[idx1, idx2] += 2 * LAMBDA

# ------------------------------------------------
# (C) 制約項2: 重さの合計を一致させる
# 数式: lambda * ( sum(k*y) - sum(w*q) )^2
# 展開 -> lambda * ( A^2 - 2AB + B^2 )
# A = sum(k*y), B = sum(w*q)
# ------------------------------------------------

# --- Part 1: A^2 (補助変数同士の積) ---
# (sum k*y)^2 = sum k^2*y + sum 2*k1*k2*y1*y2
for k1 in range(1, C + 1):
    idx1 = get_y_idx(k1)
    # 対角成分 (k^2)
    Q[idx1, idx1] += LAMBDA * (k1 ** 2)

    # 交差成分 (2 * k1 * k2)
    for k2 in range(k1 + 1, C + 1):
        idx2 = get_y_idx(k2)
        Q[idx1, idx2] += LAMBDA * 2 * k1 * k2

# --- Part 2: B^2 (商品変数同士の積) ---
# (sum w*q)^2 = sum w^2*q + sum 2*w1*w2*q1*q2
for i1 in range(N):
    idx1 = get_q_idx(i1)
    w1 = items[i1]['weight']
    # 対角成分 (w^2)
    Q[idx1, idx1] += LAMBDA * (w1 ** 2)

    # 交差成分 (2 * w1 * w2)
    for i2 in range(i1 + 1, N):
        idx2 = get_q_idx(i2)
        w2 = items[i2]['weight']
        Q[idx1, idx2] += LAMBDA * 2 * w1 * w2

# --- Part 3: -2AB (商品と補助変数の積) ---
# -2 * sum(k*y) * sum(w*q)
for k in range(1, C + 1):
    idx_y = get_y_idx(k)
    for i in range(N):
        idx_q = get_q_idx(i)
        w = items[i]['weight']

        # 係数: -2 * lambda * k * w
        coeff = -2 * LAMBDA * k * w

        # 行列の上三角成分(行 < 列)に入れるための処理
        if idx_y < idx_q:
            Q[idx_y, idx_q] += coeff
        else:
            Q[idx_q, idx_y] += coeff

###5.5.Openjijを用いて解く

In [8]:
# ==========================================
# OpenJijで解く
# ==========================================
print("最適化計算中...")
sampler = oj.SQASampler()
# num_reads: 計算回数 (多いほど安定する)
response = sampler.sample_qubo(Q, num_reads=100)

# 最良解の取得
sample = response.first.sample

# ==========================================
# 結果の表示
# ==========================================
print("\n" + "="*30)
print(" 最適化結果 ")
print("="*30)

total_w = 0
total_v = 0
active_y = 0

# --- 商品の確認 ---
print("【選ばれた商品】")
for i in range(N):
    idx = get_q_idx(i)
    if sample[idx] == 1:
        item = items[i]
        print(f"  ID:{i} (重さ:{item['weight']}, 価値:{item['value']})")
        total_w += item['weight']
        total_v += item['value']

# --- 補助変数の確認 ---
for k in range(1, C + 1):
    idx = get_y_idx(k)
    if sample[idx] == 1:
        active_y = k

print("-" * 30)
print(f"合計価値: {total_v}")
print(f"合計重量: {total_w} / 容量 {C}")
print(f"補助変数(目標重量): {active_y}")

if total_w <= C and total_w == active_y:
    print(">>> 成功: 制約を満たした最適解です。")
else:
    print(">>> 失敗: 制約違反があります (ペナルティ不足の可能性あり)。")

最適化計算中...

 最適化結果 
【選ばれた商品】
  ID:0 (重さ:2, 価値:3)
  ID:3 (重さ:5, 価値:8)
------------------------------
合計価値: 11
合計重量: 7 / 容量 7
補助変数(目標重量): 7
>>> 成功: 制約を満たした最適解です。


###5.6.解の妥当性

まず、制約条件が守られているか確認します。
* **選ばれた荷物:** ID:0 (2kg) と ID:3 (5kg)
* **合計重量:** $2 + 5 = 7$ kg
* **容量制限:** $C=7$ kg

合計重量は容量制限を超えていません（$\le 7$）。
さらに、出力にある「補助変数(目標重量): 7という点に注目してください。これは、アニーリングマシンが「荷物の合計は7kgになる」と判断し、それに対応するスラック変数 $y_7$ を正しく $1$ にしたことを意味します。
商品の重さと補助変数が完全に同期しており、数式モデル（$H_{cost}^{(2)}$）が正しく機能した証拠です。

###5.7.最適性の確認

本当にこれが一番良い組み合わせ（最適解）なのでしょうか？ 他の組み合わせと比較してみます。

* **パターンA（今回の解）:**
    * 商品0 (2kg, 価値3) + 商品3 (5kg, 価値8)
    * $\to$ 重さ 7kg, **価値 11**
* **パターンB（別の組み合わせ例）:**
    * 商品1 (3kg, 価値4) + 商品2 (4kg, 価値5)
    * $\to$ 重さ 7kg, **価値 9**
* **パターンC（軽いものをたくさん）:**
    * 商品0 + 商品1 + 商品4
    * $\to$ 重さ 6kg, **価値 8**

アニーリングマシンは、単に重さを満たすだけでなく、価値が最大になる組み合わせを正しく探索できています。